# Auditoría de Visualizaciones y Refactorización
## Pearsons_Four — Proyecto 1

### 1. Ejercicio de Autocrítica
Selecciono 3 gráficos del proyecto original por sus limitaciones:

**1. Boxplot de Seaborn (Salario por Experiencia):** Era estático, ocultaba la densidad real de las observaciones y la bimodalidad en niveles altos. No permitía identificar puntos individuales ni hacer zoom en rangos específicos.

**2. Heatmap de Seaborn (Matriz de Correlación):** La paleta `mako` no es divergente, dificultando distinguir correlaciones positivas de negativas. No se podía pasar el cursor para ver el valor exacto.

**3. Barplot de Seaborn (Salario Promedio):** Las etiquetas del eje X eran códigos numéricos sin contexto de negocio. No había interactividad para explorar subgrupos.

In [ ]:
# -----------------------------------------------------------
# CARGA DE DATOS
# -----------------------------------------------------------
# CORREGIDO: NO renombramos columnas con replace('_', ' ').
# Usamos labels={} en Plotly para cambiar SOLO cómo se muestran.
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

df = pd.read_csv('/content/linkedin_data_roles_procesed.csv')

# Mapeo a lenguaje de negocio
exp_map = {
    0: 'Prácticas', 1: 'Junior', 2: 'Asociado',
    3: 'Mid-Senior', 4: 'Director', 5: 'Ejecutivo'
}
df['exp_label'] = df['experience_level_num'].map(exp_map).fillna('No especificado')

print(f'Dataset: {df.shape[0]} filas')

FileNotFoundError: [Errno 2] No such file or directory: '/content/linkedin_data_roles_procesed (1).csv'

In [ ]:
# Función de estilo profesional
def estilo_pro(fig):
    fig.update_layout(
        width=950, height=600,
        template='plotly_white',
        margin=dict(l=60, r=60, t=80, b=60),
        font=dict(size=13)
    )
    return fig

---
### Gráfico 1: Violín Dinámico — Distribución Salarial por Experiencia
Refactoriza el boxplot de Seaborn.

**CORRECCIÓN:** Filtramos salarios > 1000 antes de aplicar escala logarítmica (log(0)=∞). Usamos `box=True` y `points='all'`.

In [ ]:
# -----------------------------------------------------------
# G1: VIOLÍN CON ESCALA LOGARÍTMICA
# -----------------------------------------------------------
df_violin = df.dropna(subset=['normalized_salary', 'exp_label']).copy()

# CORREGIDO: Filtramos salarios <= 1000 para evitar log(0)
df_violin = df_violin[df_violin['normalized_salary'] > 1000].copy()

fig1 = px.violin(
    df_violin,
    y='normalized_salary',
    x='exp_label',
    box=True,
    points='all',
    color='exp_label',
    labels={
        'normalized_salary': 'Salario Normalizado (€)',
        'exp_label': 'Nivel de Experiencia'
    },
    custom_data=['title', 'formatted_work_type', 'company_name'],
    title='<b>Distribución de Salarios por Experiencia</b><br><sub>Escala logarítmica + cuartiles + puntos individuales</sub>'
)

fig1.update_yaxes(type='log', title='Salario Normalizado (€) — Escala Logarítmica')

fig1.update_traces(
    hovertemplate='<br>'.join([
        '<b>%{customdata[0]}</b>',
        'Empresa: %{customdata[2]}',
        'Salario: €%{y:,.0f}',
        'Jornada: %{customdata[1]}',
        '<extra></extra>'
    ])
)

fig1.update_layout(showlegend=False)
estilo_pro(fig1).show()

---
### Gráfico 2: Matriz de Correlación Divergente
Refactoriza el heatmap de Seaborn.

Paleta `RdBu` divergente (rojo=positiva, azul=negativa, blanco=neutra). Anotaciones con valores exactos.

In [ ]:
# -----------------------------------------------------------
# G2: MATRIZ DE CORRELACIÓN
# -----------------------------------------------------------
cols_num = ['normalized_salary', 'experience_level_num', 'views', 'applies', 'max_salary']
corr = df[cols_num].dropna().corr()

# CORREGIDO: Renombramos TANTO columnas como índice
etiquetas = {
    'normalized_salary': 'Salario (€)',
    'experience_level_num': 'Experiencia',
    'views': 'Visualizaciones',
    'applies': 'Postulaciones',
    'max_salary': 'Salario Máx (€)'
}
corr = corr.rename(index=etiquetas, columns=etiquetas)

fig2 = px.imshow(
    corr.values,
    x=corr.columns,
    y=corr.columns,
    text_auto='.2f',
    aspect='auto',
    color_continuous_scale='RdBu',
    zmin=-1, zmax=1,
    title='<b>Matriz de Correlación de Variables</b>'
)

fig2.update_traces(
    hovertemplate='<br>'.join([
        '<b>Correlación</b>',
        'Var 1: %{y}',
        'Var 2: %{x}',
        'Coeficiente: %{z:.3f}',
        '<extra></extra>'
    ])
)

estilo_pro(fig2).show()

---
### Gráfico 3a: Barras — Salario Promedio por Experiencia
Refactoriza el barplot de Seaborn. Muestra la tendencia clara del mercado.

In [ ]:
# -----------------------------------------------------------
# G3a: BARRAS DE SALARIO PROMEDIO (fig3_bar)
# -----------------------------------------------------------
df_agrupado = df.groupby('exp_label')['normalized_salary'].mean().reset_index()

fig3_bar = px.bar(
    df_agrupado,
    x='exp_label',
    y='normalized_salary',
    text_auto='.3s',
    labels={
        'exp_label': 'Nivel de Experiencia',
        'normalized_salary': 'Salario Promedio (€)'
    },
    color='normalized_salary',
    color_continuous_scale='Blues',
    title='<b>Salario Promedio por Nivel de Experiencia</b>'
)

fig3_bar.update_traces(
    hovertemplate='<br>'.join([
        '<b>%{x}</b>',
        'Salario promedio: €%{y:,.0f}',
        '<extra></extra>'
    ])
)

fig3_bar.update_layout(showlegend=False)
estilo_pro(fig3_bar).show()

---
### Gráfico 3b: Dispersión — Salario vs Experiencia por Modalidad
Versión scatter del mismo análisis, coloreado por tipo de jornada. Útil para detectar outliers.

In [ ]:
# -----------------------------------------------------------
# G3b: DISPERSIÓN (fig3_scatter) — nombre ÚNICO, no sobrescribe
# -----------------------------------------------------------
df_scat = df.dropna(subset=['normalized_salary', 'exp_label']).copy()
df_scat = df_scat[df_scat['normalized_salary'] > 1000].copy()

fig3_scatter = px.scatter(
    df_scat,
    x='exp_label',
    y='normalized_salary',
    color='formatted_work_type',
    opacity=0.4,
    custom_data=['title', 'company_name'],
    labels={
        'normalized_salary': 'Salario Normalizado (€)',
        'exp_label': 'Nivel de Experiencia',
        'formatted_work_type': 'Modalidad'
    },
    title='<b>Salario vs Nivel de Experiencia</b><br><sub>Color por modalidad de trabajo — escala logarítmica</sub>'
)

fig3_scatter.update_yaxes(type='log')

fig3_scatter.update_traces(
    marker=dict(size=6),
    hovertemplate='<br>'.join([
        '<b>%{customdata[0]}</b>',
        'Empresa: %{customdata[1]}',
        'Salario: €%{y:,.0f}',
        '<extra></extra>'
    ])
)

fig3_scatter.update_layout(hovermode='closest')
estilo_pro(fig3_scatter).show()

---
## 4. Auditoría de Sesgos — Conclusiones

Tras explorar la interactividad de los gráficos:

**Desequilibrio de categorías:** La mayoría de los datos se concentran en niveles "Mid-Senior" y "Senior", dejando a "Junior" y "Prácticas" como minoría estadística.

**Bimodalidad en violín:** En niveles "Director" y "Ejecutivo" se ven distribuciones con picos secundarios que el boxplot tradicional oculta. Indica subgrupos salariales dentro del mismo nivel.

**Impacto en IA predictiva:** Un modelo entrenado con estos datos aprenderá un sesgo hacia salarios senior, penalizando predicciones para perfiles junior o contratos poco frecuentes. El algoritmo no entenderá la realidad del mercado, solo la sobrerrepresentación de la muestra.

---
## 5. Exportación a HTML

In [ ]:
# -----------------------------------------------------------
# EXPORTACIÓN — nombres ÚNICOS para cada archivo
# -----------------------------------------------------------
fig1.write_html('01_violin_salarios.html')
fig2.write_html('02_matriz_correlacion.html')
fig3_bar.write_html('03_barras_salario_promedio.html')
fig3_scatter.write_html('04_scatter_salario_vs_experiencia.html')

print('Archivos exportados correctamente:')
import os
for f in ['01_violin_salarios.html', '02_matriz_correlacion.html',
          '03_barras_salario_promedio.html', '04_scatter_salario_vs_experiencia.html']:
    if os.path.exists(f):
        print(f'  ✅ {f}')
    else:
        print(f'  ❌ {f} — NO ENCONTRADO')